# 14. Time-aware Market Unit-price Correction

Residual correction이 한계에 가까워졌기 때문에, 14는 과거 거래만 사용한 시세 기반 보정을 추가합니다.

- Base: 09 재현 + 11 `Residual09` price-brand correction
- Market signal: 검증 시점보다 과거인 거래의 `Gu × Age_Bin`별 smoothed ㎡당가
- Prediction: `market_price = smoothed_unit_price × Exclusive_Area`
- Correction: `pred14 = pred11 + 0.03 × (market_price - pred11)`
- Gate: 이전 OOF 기준 `pred_range`가 낮은 안정 샘플에만 적용
- Test는 최종 단계에서 Train으로 fit한 market map의 transform 대상으로만 사용합니다.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

# 09의 검증/최종 예측 로직을 재현합니다.
# 이후 11 보정과 14 market correction만 추가합니다.
nb09 = json.loads((PROJECT_ROOT / 'notebooks' / '09_residual08_cluster_correction.ipynb').read_text())
code09 = ''.join(nb09['cells'][1]['source'])
namespace = {}
exec(code09, namespace)

rmse = namespace['rmse']
rmsle = namespace['rmsle']
smoothed_residual_prediction = namespace['smoothed_residual_prediction']
oof_frames = namespace['oof_frames']
TIME_FOLDS = namespace['TIME_FOLDS']
train = namespace['train']
test = namespace['test']
residual_train = namespace['residual_train'].copy()
test_residual_frame = namespace['test_residual_frame'].copy()
test_pred_09 = namespace['test_pred'].copy()
sample_submission = namespace['sample_submission']
SUBMISSION_DIR = namespace['SUBMISSION_DIR']

PRICE_BRAND_PRIOR = 10
PRICE_BRAND_ALPHA = 0.20
MARKET_GROUP = 'Gu_AgeBin'
MARKET_PRIOR = 100
MARKET_ALPHA = 0.03

def add_price_brand_group(df):
    x = df.copy()
    x['BasePred_Brand'] = x['BasePred_Bin'].astype(str) + '_' + x['Brand_Apartment'].astype(str)
    return x

def add_market_bins(df):
    x = df.copy().reset_index(drop=True)
    year = x['Transaction_YearMonth'] // 100
    x['Age_at_Transaction'] = year - x['Year_Built']
    x['Age_Bin'] = pd.cut(
        x['Age_at_Transaction'], bins=[-10, 5, 15, 25, 40, 200],
        labels=False, include_lowest=True,
    ).astype('int16')
    x['Gu_AgeBin'] = x['Gu'].astype(str) + '_' + x['Age_Bin'].astype(str)
    if 'Target' in x.columns:
        x['UnitPrice'] = x['Target'] / x['Exclusive_Area']
    return x

def fit_market_unit_price_predict(fit_df, predict_df, group_col=MARKET_GROUP, prior=MARKET_PRIOR):
    fit_raw = add_market_bins(fit_df)
    pred_raw = add_market_bins(predict_df)
    global_unit_price = fit_raw['UnitPrice'].mean()
    stats = fit_raw.groupby(group_col)['UnitPrice'].agg(['sum', 'count'])
    mapping = (stats['sum'] + prior * global_unit_price) / (stats['count'] + prior)
    unit_price = pred_raw[group_col].map(mapping).fillna(global_unit_price).to_numpy()
    return unit_price * pred_raw['Exclusive_Area'].to_numpy()

# OOF: 09 -> 11 -> 14
history_11 = []
history_14 = []
fold_rows = []
oof_true, oof_09, oof_11, oof_14 = [], [], [], []

for (fold_name, valid_months), frame in zip(TIME_FOLDS, oof_frames):
    valid_df = train.loc[train['Transaction_YearMonth'].isin(valid_months)].copy()
    fit_df = train.loc[train['Transaction_YearMonth'] < valid_months.min()].copy()
    frame = add_price_brand_group(frame.copy())
    frame['Residual09'] = frame['Target'] - frame['Pred09']

    y = valid_df['Target'].to_numpy()
    pred09 = frame['Pred09'].to_numpy()
    assert len(y) == len(pred09)

    if int(frame['Fold'].iloc[0]) == 1:
        pred11 = pred09.copy()
        pred14 = pred11.copy()
        applied11 = 0
        applied14 = 0
    else:
        local_train = pd.concat(history_11, ignore_index=True)
        local_residual = smoothed_residual_prediction(
            local_train, frame, 'BasePred_Brand', 'Residual09', PRICE_BRAND_PRIOR
        )
        mask11 = frame['base_pred'].to_numpy() <= local_train['base_pred'].quantile(0.80)
        adjustment11 = np.zeros(len(frame))
        adjustment11[mask11] = local_residual[mask11]
        pred11 = np.clip(pred09 + PRICE_BRAND_ALPHA * adjustment11, 0, None)
        applied11 = int(mask11.sum())

        market_price = fit_market_unit_price_predict(fit_df, valid_df)
        market_train = pd.concat(history_14, ignore_index=True)
        mask14 = frame['pred_range'].to_numpy() <= market_train['pred_range'].median()
        pred14 = pred11.copy()
        pred14[mask14] = np.clip(
            pred11[mask14] + MARKET_ALPHA * (market_price[mask14] - pred11[mask14]),
            0, None,
        )
        applied14 = int(mask14.sum())

    frame['Pred11'] = pred11
    frame['Pred14'] = pred14
    history_11.append(frame)
    history_14.append(frame)

    fold_rows.append({
        'Fold': int(frame['Fold'].iloc[0]),
        'Rows': len(frame),
        '09_RMSE': rmse(y, pred09),
        '11_RMSE': rmse(y, pred11),
        '14_RMSE': rmse(y, pred14),
        '14_RMSLE': rmsle(y, pred14),
        '11_vs_09_Improvement': rmse(y, pred09) - rmse(y, pred11),
        '14_vs_11_Improvement': rmse(y, pred11) - rmse(y, pred14),
        'Applied_11_Rows': applied11,
        'Applied_14_Rows': applied14,
        'Mean_14_Move': np.mean(np.abs(pred14 - pred11)),
    })
    oof_true.extend(y)
    oof_09.extend(pred09)
    oof_11.extend(pred11)
    oof_14.extend(pred14)

scores = pd.DataFrame(fold_rows)
display(scores)

pooled_09 = rmse(oof_true, oof_09)
pooled_11 = rmse(oof_true, oof_11)
pooled_14 = rmse(oof_true, oof_14)
eligible = scores.loc[scores['Fold'] > 1].copy()
improved_folds = int((eligible['14_RMSE'] < eligible['11_RMSE']).sum())
min_improvement = eligible['14_vs_11_Improvement'].min()
pooled_improvement = pooled_11 - pooled_14

print(f'09 pooled RMSE: {pooled_09:,.2f}')
print(f'11 pooled RMSE: {pooled_11:,.2f}')
print(f'14 pooled RMSE: {pooled_14:,.2f}')
print(f'14 vs 11 pooled improvement: {pooled_improvement:,.2f}')
print(f'Improved folds: {improved_folds}/{len(eligible)}')

assert improved_folds == len(eligible)
assert pooled_improvement > 3.0
assert min_improvement > 3.0

# Final Test prediction: 09 -> 11 -> 14.
residual_train_11 = add_price_brand_group(residual_train.copy())
residual_train_11['Residual09'] = residual_train_11['Target'] - residual_train_11['Pred09']
test_residual_frame_11 = add_price_brand_group(test_residual_frame.copy())

local_residual_test = smoothed_residual_prediction(
    residual_train_11, test_residual_frame_11, 'BasePred_Brand', 'Residual09', PRICE_BRAND_PRIOR
)
mask11_test = test_residual_frame_11['base_pred'].to_numpy() <= residual_train_11['base_pred'].quantile(0.80)
adjustment11_test = np.zeros(len(test_residual_frame_11))
adjustment11_test[mask11_test] = local_residual_test[mask11_test]
test_pred_11 = np.clip(test_pred_09 + PRICE_BRAND_ALPHA * adjustment11_test, 0, None)

market_price_test = fit_market_unit_price_predict(train, test)
market_range_threshold = residual_train_11['pred_range'].median()
mask14_test = test_residual_frame_11['pred_range'].to_numpy() <= market_range_threshold
test_pred = test_pred_11.copy()
test_pred[mask14_test] = np.clip(
    test_pred_11[mask14_test] + MARKET_ALPHA * (market_price_test[mask14_test] - test_pred_11[mask14_test]),
    0, None,
)

leakage_audit = pd.Series({
    'Market unit-price maps fitted on past Train/fold-fit only': True,
    'Validation market features use only transactions earlier than validation months': True,
    'Final Test market map fitted on full Train only': True,
    'Correction alpha and gate selected from Train OOF only': True,
    'Test limited to transform and predict': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred})
submission = sample_submission[['ID']].merge(prediction_frame, on='ID', how='left', validate='one_to_one')
assert submission['Target'].notna().all()
assert len(submission) == len(sample_submission)
assert sample_submission['ID'].equals(submission['ID'])

output_path = SUBMISSION_DIR / '14_time_aware_market_features_submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
display(submission.head())


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,09_RMSE,09_RMSLE,08_vs_06_Improvement,09_vs_08_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows,Third_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538","2,614.984309",0.080279,3.177802,3.067229,141,103,108
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861","2,516.302312",0.071534,2.237174,3.842550,146,100,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
09 pooled RMSE: 2,410.15
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
Third-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84
09 vs 08 pooled improvement: 2.34


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Third local correction selected with strict Train OOF thresholds     True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/09_residual08_cluster_correction_submission.csv


...

,Fold,Rows,09_RMSE,11_RMSE,14_RMSE,14_RMSLE,11_vs_09_Improvement,14_vs_11_Improvement,Applied_11_Rows,Applied_14_Rows,Mean_14_Move
0,1,278,"2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0.000000
1,2,244,"2,614.984309","2,609.888396","2,600.182719",0.080002,5.095913,9.705677,193,108,78.068367
2,3,246,"2,516.302312","2,514.519354","2,510.247782",0.071348,1.782957,4.271573,177,100,65.808296


09 pooled RMSE: 2,410.15
11 pooled RMSE: 2,407.79
14 pooled RMSE: 2,403.03
14 vs 11 pooled improvement: 4.77
Improved folds: 2/2


Market unit-price maps fitted on past Train/fold-fit only                          True
Validation market features use only transactions earlier than validation months    True
Final Test market map fitted on full Train only                                    True
Correction alpha and gate selected from Train OOF only                             True
Test limited to transform and predict                                              True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/14_time_aware_market_features_submission.csv


,ID,Target
0,TR_2427,"36,892.186265"
1,TR_0028,"48,282.090800"
2,TR_1461,"29,192.342366"
3,TR_1977,"47,499.996303"
4,TR_2351,"47,156.525211"


## 실험 결론

14는 residual correction만 반복하던 흐름에서 벗어나, 검증 시점보다 과거인 거래의 지역×연식 단가를 시세 신호로 사용합니다. 단가 기반 market estimate는 11 예측을 아주 약하게 보정하며, Train OOF에서 Fold 2와 Fold 3 모두 11 대비 충분히 개선될 때만 제출 파일을 생성합니다.
